# Baseline: No Augmentation (ImageNet-100)

This notebook mirrors `01_baseline` but uses the **CSS (CSV-saving + per-prediction) logging logic** from notebooks `11` and `07`, with augmentation fixed to `no_augmentation`.

Purpose: produce a clean baseline row in `lbp_outputs/runs.csv` and `predictions.csv` so results can be compared side-by-side with the augmented hybrid runs.

Models covered:
- DeiT-Tiny, DeiT-Small, DeiT-Base
- ViT-Tiny, ViT-Small, ViT-Base

## 1 · Setup

In [ ]:
# Run once per fresh Colab runtime
!git clone https://github.com/Chalhotra/ViT-Token-Economy.git
%cd ViT-Token-Economy

In [ ]:
!git checkout priyanshu/augmentation

In [ ]:
!pip -q install -r requirements.txt
!pip -q install -e .

## 2 · Imports

In [ ]:
from augmentation import no_augmentation
from src.imagenet_mapping import build_imagenet100_to_1k_map
from src.models import ModelConfig, create_model, shrink_imagenet1k_head_to_imagenet100
from src.data import DataConfig, load_imagenet100_split, build_transform_for_model, apply_timm_preprocess, build_loader, make_collate_fn
from src.eval import evaluate_accuracy_latency_throughput, evaluate_with_topk_predictions, compute_gflops
from src.utils import get_device, num_params
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
from IPython.display import display

In [ ]:
device = get_device()
maps = build_imagenet100_to_1k_map()
print(f'Using device: {device}')

## 3 · CSS Output Setup

Mirrors the `lbp_outputs` logging logic from notebooks 11 & 07.

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lbp_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_CSV        = OUTPUT_DIR / 'runs.csv'
PREDICTIONS_CSV = OUTPUT_DIR / 'predictions.csv'

RUNS_COLUMNS = [
    'run_id', 'model_id', 'augmentation', 'aug_params', 'efficiency_budget',
    'profile', 'config_key', 'acc1', 'gflops', 'latency_ms', 'params_m',
    'throughput', 'timestamp',
]
PREDICTIONS_COLUMNS = [
    'run_id', 'image_id', 'ground_truth_label', 'rank', 'predicted_class',
    'confidence', 'is_correct', 'entropy', 'conf_gap_to_rank1',
]

def _append_csv_rows(csv_path, rows, dedupe_subset):
    """Append rows to a CSV, deduplicating on the given subset of columns."""
    frame = pd.DataFrame(rows)
    if csv_path.exists():
        existing = pd.read_csv(csv_path)
        frame = pd.concat([existing, frame], ignore_index=True)
        frame = frame.drop_duplicates(subset=dedupe_subset, keep='last')
    frame.to_csv(csv_path, index=False)

if 'results' not in globals():
    results = []

print(f'Output directory: {OUTPUT_DIR}')
print(f'Runs CSV        : {RUNS_CSV}')
print(f'Predictions CSV : {PREDICTIONS_CSV}')

## 4 · Core Baseline Run Function

Combines the simple `run_baseline_test` from `01_baseline` with the CSS logging (`_append_csv_rows`) and per-prediction collection from notebooks 11 & 07.
Augmentation is always `no_augmentation` so rows are directly comparable to baseline.

In [ ]:
def run_baseline_test(
    model_id: str,
    batch_size: int = 64,
    collect_predictions: bool = True,
):
    aug_name    = 'no_augmentation'
    aug_builder = no_augmentation
    config_key  = 'baseline'
    run_id      = f'run_{model_id}__baseline__{aug_name}'

    print('\n' + '=' * 80)
    print(f'Testing baseline model : {model_id}')
    print(f'Augmentation           : {aug_name}')
    print('=' * 80 + '\n')

    model = create_model(ModelConfig(model_id=model_id, pretrained=True))
    model = shrink_imagenet1k_head_to_imagenet100(model, maps.new_to_old_map, num_classes=100)
    model = model.to(device).eval()

    print('➡️ Loading dataset...')
    ds = load_imagenet100_split(DataConfig(split='validation'))
    class_names = (
        ds.features['label'].names
        if hasattr(ds.features['label'], 'names')
        else [str(i) for i in range(100)]
    )
    print('➡️ Building transform...')
    transform = build_transform_for_model(model)
    print('➡️ Building augmentation pipeline (no_augmentation)...')
    aug_pipeline = aug_builder() if callable(aug_builder) else aug_builder
    print('➡️ Applying preprocess...')
    ds_t, transform = apply_timm_preprocess(ds, transform, aug_pipeline=aug_pipeline)
    print('➡️ Creating collate_fn...')
    collate_fn = make_collate_fn(transform)
    print('➡️ Building loader...')
    loader = build_loader(
        ds_t,
        DataConfig(batch_size=batch_size, split='validation', shuffle=False),
        collate_fn=collate_fn,
    )

    print('➡️ Starting evaluation...')
    if collect_predictions:
        print('Storing top-10 predictions per image...')
        metrics, prediction_rows = evaluate_with_topk_predictions(
            model, loader, device, class_names=class_names, topk=10
        )
    else:
        metrics = evaluate_accuracy_latency_throughput(model, loader, device)
        prediction_rows = []

    sample = transform(ds_t[0]['pixel_values']).unsqueeze(0).to(device)
    print('➡️ Computing GFLOPs...')
    gflops = compute_gflops(model, sample)

    timestamp = datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
    aug_params = json.dumps({'batch_size': batch_size, 'augmentation': aug_name}, sort_keys=True)

    result = {
        'run_id'            : run_id,
        'model_id'          : model_id,
        'model'             : model_id,
        'config_name'       : f'Baseline {model_id}',
        'augmentation'      : aug_name,
        'aug_params'        : aug_params,
        'efficiency_budget' : 1.0,   # no token reduction
        'profile'           : 'baseline',
        'config_key'        : config_key,
        'params_m'          : num_params(model) / 1e6,
        'gflops'            : gflops,
        'timestamp'         : timestamp,
        **metrics,
    }

    print(f"Results for {model_id}:")
    print(f"  Top-1 Accuracy : {metrics['acc1']:.2f}%")
    print(f"  GFLOPs         : {gflops:.3f}")
    print(f"  Latency        : {metrics['latency_ms']:.2f} ms")
    print(f"  Throughput     : {metrics['throughput']:.1f} samples/sec")

    # --- CSS: persist run summary ---
    run_record = {col: result.get(col) for col in RUNS_COLUMNS}
    _append_csv_rows(RUNS_CSV, [run_record], dedupe_subset=['run_id'])

    # --- CSS: persist per-prediction rows ---
    prediction_records = []
    for row in prediction_rows:
        row = dict(row)
        row['run_id'] = run_id
        prediction_records.append({col: row.get(col) for col in PREDICTIONS_COLUMNS})
    if prediction_records:
        _append_csv_rows(PREDICTIONS_CSV, prediction_records,
                         dedupe_subset=['run_id', 'image_id', 'rank'])

    print(f'Saved run summary  → {RUNS_CSV}')
    if prediction_records:
        print(f'Saved predictions  → {PREDICTIONS_CSV}')
    return result

## 5 · Per-Model Run Cells

Run each cell independently. Results accumulate in `results` and are persisted to CSVs immediately.

In [ ]:
# Baseline (no augmentation): DeiT-Tiny
if 'results' not in globals():
    results = []

name = 'deit_tiny_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

In [ ]:
# Baseline (no augmentation): DeiT-Small
if 'results' not in globals():
    results = []

name = 'deit_small_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

In [ ]:
# Baseline (no augmentation): DeiT-Base
if 'results' not in globals():
    results = []

name = 'deit_base_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

In [ ]:
# Baseline (no augmentation): ViT-Tiny
if 'results' not in globals():
    results = []

name = 'vit_tiny_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

In [ ]:
# Baseline (no augmentation): ViT-Small
if 'results' not in globals():
    results = []

name = 'vit_small_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

In [ ]:
# Baseline (no augmentation): ViT-Base
if 'results' not in globals():
    results = []

name = 'vit_base_patch16_224'
results = [x for x in results if x.get('model') != name]
results.append(run_baseline_test(name))

## 6 · Final Comparison Table

In [ ]:
# Comparison table — all models run so far
if 'results' not in globals() or not results:
    print('No results yet. Run one or more model cells above first.')
else:
    df = pd.DataFrame(results).copy()
    for col in ['acc1', 'latency_ms', 'throughput', 'gflops', 'params_m']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').round(3)
    df = df.sort_values('model').reset_index(drop=True)
    cols = [c for c in ['model', 'acc1', 'latency_ms', 'throughput', 'gflops', 'params_m'] if c in df.columns]
    display(df[cols])

## 7 · Load Previously Saved Results from CSV

If the runtime was reset, reload results from the persisted CSVs.

In [ ]:
if RUNS_CSV.exists():
    df_saved = pd.read_csv(RUNS_CSV)
    # Filter to baseline-only rows
    df_baseline_saved = df_saved[df_saved['profile'] == 'baseline'].copy()
    print(f'Loaded {len(df_baseline_saved)} baseline run(s) from {RUNS_CSV}')
    cols = [c for c in ['model_id', 'acc1', 'gflops', 'latency_ms', 'throughput', 'params_m', 'augmentation', 'timestamp'] if c in df_baseline_saved.columns]
    display(df_baseline_saved[cols])
else:
    print('No runs.csv found yet. Run the model cells above first.')